# Dự án GNN · Cạnh có giúp phân loại nút?
Mỗi nút là một bài báo giả lập. Chủ đề ẩn chi phối cả vector từ ngữ có nhiễu và xác suất liên kết. Nhãn để chấm là chủ đề, nhưng cột nhãn không được đưa vào đầu vào của mô hình. Dữ liệu có tín hiệu đồng chủ đề được thiết kế để thử cơ chế; kết quả không suy rộng trực tiếp sang mạng trích dẫn thực.

**A.** Dự đoán trước mô hình nào sẽ hưởng lợi từ cạnh và vì sao. Ta chia theo đồ thị độc lập, không có cạnh giữa các tập. Validation chọn checkpoint; test chỉ chạy sau khi chọn.

MLP và GCN chỉ giữ self-loop có cùng phép tính trong thiết lập này. Giữ cả hai như một phép đối chứng: kết quả của chúng phải giống nhau khi dùng cùng khởi tạo. GCN dùng hai phép truyền tin và cùng đầu phân loại hai lớp như MLP.

Nguồn đọc tiếp: Liu & Zhou, §5.1.3 và Ch. 15.

In [ ]:
import json
from pathlib import Path
import numpy as np

NODES = 36
CLASSES = 3
SEEDS = (7, 19, 42)
MODELS = ("MLP", "GCN", "GCN tự nối")
TRAIN, VALIDATION, TEST = slice(0, 12), slice(12, 16), slice(16, 20)

def make_graph(seed):
    rng = np.random.default_rng(seed)
    topic = rng.permutation(np.repeat(np.arange(CLASSES), NODES // CLASSES))
    means = np.array([[.75, -.2], [-.2, .75], [-.55, -.55]])
    features = rng.normal(size=(NODES, 2)) + means[topic]
    probability = np.where(topic[:, None] == topic[None, :], .32, .055)
    upper = np.triu(rng.random((NODES, NODES)) < probability, 1)
    adjacency = np.eye(NODES) + upper + upper.T
    root = np.sqrt(adjacency.sum(axis=1))
    propagation = adjacency / root[:, None] / root[None, :]
    return features, propagation, topic

def split_data(seed):
    children = np.random.SeedSequence(seed).spawn(20)
    graphs = [make_graph(child) for child in children]
    split_ids = [set(range(20)[subset]) for subset in (TRAIN, VALIDATION, TEST)]
    assert len(set.union(*split_ids)) == sum(map(len, split_ids)) == 20
    return tuple(zip(*graphs))

def scores(y, pred):
    confusion = np.zeros((CLASSES, CLASSES), dtype=int)
    np.add.at(confusion, (y.ravel(), pred.ravel()), 1)
    denominator = confusion.sum(axis=1) + confusion.sum(axis=0)
    per_class = np.divide(2 * confusion.diagonal(), denominator,
                          out=np.zeros(CLASSES), where=denominator > 0)
    return float(per_class.mean()), float((pred == y).mean()), confusion.tolist()

In [ ]:
def forward(x, propagation, params):
    w1, b1, w2, b2 = params
    incoming = propagation @ x
    hidden = np.tanh(incoming @ w1 + b1)
    combined = propagation @ hidden
    return incoming, hidden, combined, combined @ w2 + b2

def train_each(features, propagation, labels, seed):
    rng = np.random.default_rng(seed)
    params = [rng.normal(0, .3, (2, 12)), np.zeros(12),
              rng.normal(0, .3, (12, CLASSES)), np.zeros(CLASSES)]
    x, p, y = features[TRAIN], propagation[TRAIN], labels[TRAIN]
    best_score, best_epoch, best = -1., 0, None
    for epoch in range(1, 121):
        incoming, hidden, combined, logits = forward(x, p, params)
        logits -= logits.max(axis=-1, keepdims=True)
        probs = np.exp(logits)
        probs /= probs.sum(axis=-1, keepdims=True)
        gradient = (probs - np.eye(CLASSES)[y]) / y.size
        grad_hidden = (p.transpose(0, 2, 1) @ (gradient @ params[2].T)) * (1 - hidden**2)
        gradients = [incoming.reshape(-1, 2).T @ grad_hidden.reshape(-1, 12),
                     grad_hidden.sum(axis=(0, 1)),
                     combined.reshape(-1, 12).T @ gradient.reshape(-1, CLASSES),
                     gradient.sum(axis=(0, 1))]
        params = [pval - .35 * grad for pval, grad in zip(params, gradients)]
        val_predictions = forward(features[VALIDATION], propagation[VALIDATION], params)[-1].argmax(-1)
        val_f1 = scores(labels[VALIDATION], val_predictions)[0]
        if val_f1 > best_score:
            best_score, best_epoch = val_f1, epoch
            best = [param.copy() for param in params]
    predictions = forward(features[TEST], propagation[TEST], best)[-1].argmax(-1)
    f1, acc, cm = scores(labels[TEST], predictions)
    return dict(macro_f1=f1, accuracy=acc, confusion=cm, validation_f1=best_score, best_epoch=best_epoch)

def experiment(seeds=SEEDS):
    runs = []
    for seed in seeds:
        x, graph, y = map(np.asarray, split_data(seed))
        identity = np.broadcast_to(np.eye(NODES), graph.shape)
        rows = [dict(model=model, **train_each(x, matrix, y, seed))
                for model, matrix in zip(MODELS, (identity, graph, identity))]
        assert rows[0]["confusion"] == rows[2]["confusion"]
        assert all(sum(map(sum, row["confusion"])) == 4 * NODES for row in rows)
        runs.append(dict(seed=seed, models=rows))
    summary = [dict(model=name, **{metric: float(np.mean([run["models"][i][metric] for run in runs]))
               for metric in ("macro_f1", "accuracy")}) for i, name in enumerate(MODELS)]
    return dict(runs=runs, summary=summary)

result = experiment()
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

## B và C · Giải thích kết quả
**B.** Chọn một seed. Đọc ma trận nhầm lẫn (hàng là nhãn thật, cột là nhãn đoán) và nêu một nhóm lỗi thay đổi khi bổ sung cạnh. Cắt nghĩa vì sao dùng đúng cùng test và cùng seed giúp so công bằng hơn.

**C.** Tạo một thí nghiệm mới trước khi xem test: giảm xác suất nối cùng chủ đề, hoặc đổi quy tắc nối để các bài khác chủ đề dễ nối với nhau hơn. Lưu báo cáo mới và giải thích liệu lợi thế GCN còn tồn tại.

Kết quả trên web lấy từ JSON của lần chạy với thiết lập mặc định. Việc sửa notebook không tự cập nhật trang web.

In [ ]:
for run in result["runs"]:
    print("Seed", run["seed"])
    for row in run["models"]:
        print(row["model"], ":", np.array(row["confusion"]).tolist())
# Bỏ chú thích để xuất kết quả của lần chạy.
# Path("GNN_ket_qua.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")